###Initialization of libraries and path

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import *
from pyspark.sql.types import *

from datetime import datetime

CATALOG = "quickcart"

SOURCE_VOLUME = f"/Volumes/{CATALOG}/default/source_data"

BRONZE_SCHEMA = "bronze"

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS quickcart.bronze;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS quickcart.bronze.bronze_ingestion_audit(
    source_table STRING,
    target_table STRING,
    start_time TIMESTAMP,
    end_time TIMESTAMP,
    duration_seconds DOUBLE,
    status STRING,
    error_message STRING,
    record_count BIGINT
) USING DELTA;

In [0]:
%sql
SELECT *
FROM quickcart.bronze.bronze_ingestion_audit;

###Building the function for ingestion

In [0]:
def ingest_to_bronze(source_table, source_format = 'parquet'):
    source_path = f"{SOURCE_VOLUME}/{source_table}"
    schema_path = f"{SOURCE_VOLUME}/_schemas/{source_table}"
    checkpoint_path = f"{SOURCE_VOLUME}/_checkpoints/{source_table}"
    target_table = f"{CATALOG}.{BRONZE_SCHEMA}.{source_table}"

    start_time = datetime.now()

    print("=" * 70)
    print(f"Starting Bronze ingestion: {source_table}")
    print("=" * 70)
    
    print(f"Source Path     : {source_path}")
    print(f"Schema Location : {schema_path}")
    print(f"Checkpoint      : {checkpoint_path}")
    print(f"Target Table    : {target_table}")

    audit_schema = StructType([
    StructField(
        "source_table",
        StringType(),
        True
    ),
    StructField(
        "target_table",
        StringType(),
        True
    ),
    StructField(
        "start_time",
        TimestampType(),
        True
    ),
    StructField(
        "end_time",
        TimestampType(),
        True
    ),
    StructField(
        "duration_seconds",
        DoubleType(),
        True
    ),
    StructField(
        "status",
        StringType(),
        True
    ),
    StructField(
        "error_message",
        StringType(),
        True
    ),
    StructField(
        "record_count",
        LongType(),
        True
    )
    ])

    try:
            df = spark.readStream.format("cloudFiles")\
                .option("cloudFiles.format",source_format)\
                    .option("cloudFiles.schemaLocation",schema_path)\
                        .load(source_path)

            df = (df.withColumn("_ingestion_timestamp",F.current_timestamp())\
                .withColumn("_source_file",F.col("_metadata.file_path"))          
                )
            
            query = df.writeStream.format("delta")\
                .outputMode("append")\
                .option("checkpointLocation", checkpoint_path)\
                    .trigger(availableNow = True)\
                        .toTable(target_table)
            
            query.awaitTermination()

            end_time = datetime.now()
            duration = (end_time-start_time).total_seconds()

            record_count = spark.sql(
            f"""
            SELECT COUNT(*) AS count
            FROM {target_table}
            """
            ).collect()[0]["count"]

            audit_data = [
            (
                source_table,
                target_table,
                start_time,
                end_time,
                duration,
                "SUCCESS",
                None,
                record_count
            )
            ]

            audit_df = spark.createDataFrame(
            audit_data,
            audit_schema
            )

            audit_df.write.mode("append").saveAsTable(
            "quickcart.bronze.bronze_ingestion_audit"
            )

            print(
            f"SUCCESS: {source_table} | "
            f"Records: {record_count} | "
            f"Duration: {duration:.2f} sec"
            )

            return "SUCCESS"

    except Exception as e:
            end_time = datetime.now()

            duration = (
                end_time - start_time
            ).total_seconds()

            error_message = str(e)
            
            audit_data = [
            (
                source_table,
                target_table,
                start_time,
                end_time,
                duration,
                "FAILED",
                error_message,
                0
            )
            ]

            audit_df = spark.createDataFrame(
            audit_data,
            audit_schema
            )

            audit_df.write.mode("append").saveAsTable(
            "quickcart.bronze.bronze_ingestion_audit"
            )

            print(
            f"FAILED: {source_table}"
            )

            print(
            f"Error: {error_message}"
            )

            return "FAILED"
            

In [0]:
ingest_to_bronze("unknown_table")

In [0]:
%sql
SELECT * FROM quickcart.bronze.payments
LIMIT 10;